In [1]:
import os
import sys
import time
import joblib
import pandas as pd
import numpy as np
import yaml
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
from modeling import (build_model_pipeline, make_objective, tune_threshold,
                       bootstrap_ci, evaluate_at_threshold)

with open(os.path.join(PROJECT_ROOT, "config", "config.yaml")) as f:
    config = yaml.safe_load(f)

SEED = config["seed"]
PROCESSED_DIR = os.path.join(PROJECT_ROOT, config["paths"]["processed_dir"])
MODELS_DIR = os.path.join(PROJECT_ROOT, config["paths"]["models_dir"], "pakistan")
TABLES_DIR = os.path.join(PROJECT_ROOT, config["paths"]["tables_dir"])
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

N_TRIALS_TREE = config["optuna"]["n_trials_tree"]
N_TRIALS_SVM = config["optuna"]["n_trials_svm"]
SVM_FIT_ROWS = config["svm_subsample"]["fit_rows"]
N_BOOTSTRAP = config["bootstrap"]["n_resamples"]

In [2]:
data_dir = os.path.join(PROCESSED_DIR, "pakistan")
train = pd.read_csv(os.path.join(data_dir, "train.csv"))
cal = pd.read_csv(os.path.join(data_dir, "calibration.csv"))
test = pd.read_csv(os.path.join(data_dir, "test.csv"))

LABEL = "y_depression"
FEATURE_COLS = [c for c in train.columns if c != LABEL]
NUMERIC_COLS = ["Age", "Gestational Age", "Number of sons ", "Number of daughters",
                "Total Number of Children"]
CATEGORICAL_COLS = [c for c in FEATURE_COLS if c not in NUMERIC_COLS]

X_train, y_train = train[FEATURE_COLS], train[LABEL]
X_cal, y_cal = cal[FEATURE_COLS], cal[LABEL]
X_test, y_test = test[FEATURE_COLS], test[LABEL]

from preprocessing import compute_group_id
groups_train = compute_group_id(X_train, FEATURE_COLS)

print("Train/Cal/Test shapes:", X_train.shape, X_cal.shape, X_test.shape)
print("Unique feature combos in training set:", groups_train.nunique(), "of", len(X_train))

Train/Cal/Test shapes: (8402, 16) (2808, 16) (2798, 16)
Unique feature combos in training set: 4191 of 8402


In [3]:
MODEL_NAMES = ["dt", "rf", "gb", "svm", "nb", "knn", "xgb"]

TRIAL_BUDGET = {
    "dt": N_TRIALS_TREE, "rf": N_TRIALS_TREE, "gb": N_TRIALS_TREE, "xgb": N_TRIALS_TREE,
    "svm": N_TRIALS_SVM, "knn": N_TRIALS_SVM, "nb": N_TRIALS_SVM,
}
print(TRIAL_BUDGET)

{'dt': 30, 'rf': 30, 'gb': 30, 'xgb': 30, 'svm': 15, 'knn': 15, 'nb': 15}


In [4]:
best_params = {}
best_cv_scores = {}

for model_name in MODEL_NAMES:
    print(f"\n{'='*50}\nTuning: {model_name}\n{'='*50}")
    t0 = time.time()

    subsample_n = SVM_FIT_ROWS if model_name == "svm" else None

    objective = make_objective(model_name, X_train, y_train, NUMERIC_COLS, CATEGORICAL_COLS,
                                seed=SEED, n_splits=5, subsample_n=subsample_n,
                                groups=groups_train)

    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=TRIAL_BUDGET[model_name], show_progress_bar=False)

    best_params[model_name] = study.best_params
    best_cv_scores[model_name] = study.best_value

    print(f"Best CV AUC: {study.best_value:.4f}  |  Time: {time.time()-t0:.1f}s")
    print(f"Best params: {study.best_params}")


Tuning: dt
Best CV AUC: 0.5294  |  Time: 113.2s
Best params: {'max_depth': 2, 'min_samples_split': 39, 'min_samples_leaf': 17, 'criterion': 'gini'}

Tuning: rf
Best CV AUC: 0.5378  |  Time: 312.6s
Best params: {'n_estimators': 574, 'max_depth': 3, 'min_samples_split': 36, 'min_samples_leaf': 16, 'max_features': 'log2'}

Tuning: gb
Best CV AUC: 0.5415  |  Time: 1376.4s
Best params: {'n_estimators': 240, 'learning_rate': 0.19024182447263455, 'max_depth': 9, 'subsample': 0.8026553961682419, 'min_samples_leaf': 5}

Tuning: svm


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

Best CV AUC: 0.5452  |  Time: 3033.3s
Best params: {'C': 21.368329072358772, 'gamma': 0.0007068974950624604}

Tuning: nb
Best CV AUC: 0.5276  |  Time: 47.4s
Best params: {'var_smoothing': 1.7670169402947963e-10}

Tuning: knn
Best CV AUC: 0.5288  |  Time: 52.0s
Best params: {'n_neighbors': 25, 'weights': 'distance', 'p': 1}

Tuning: xgb
Best CV AUC: 0.5380  |  Time: 193.1s
Best params: {'n_estimators': 425, 'learning_rate': 0.020589728197687916, 'max_depth': 3, 'subsample': 0.5917022549267169, 'colsample_bytree': 0.6521211214797689, 'min_child_weight': 11}


In [5]:
import json
with open(os.path.join(MODELS_DIR, "best_hyperparameters.json"), "w") as f:
    json.dump(best_params, f, indent=2)
print("Saved best_hyperparameters.json")

for name, score in best_cv_scores.items():
    print(f"{name}: CV AUC = {score:.4f}")

Saved best_hyperparameters.json
dt: CV AUC = 0.5294
rf: CV AUC = 0.5378
gb: CV AUC = 0.5415
svm: CV AUC = 0.5452
nb: CV AUC = 0.5276
knn: CV AUC = 0.5288
xgb: CV AUC = 0.5380


In [6]:
fitted_pipelines = {}

for model_name in MODEL_NAMES:
    pipe = build_model_pipeline(model_name, best_params[model_name],
                                 NUMERIC_COLS, CATEGORICAL_COLS, seed=SEED)
    pipe.fit(X_train, y_train)
    fitted_pipelines[model_name] = pipe
    print(f"Fitted: {model_name}")

Fitted: dt
Fitted: rf
Fitted: gb


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Fitted: svm
Fitted: nb
Fitted: knn
Fitted: xgb


In [7]:
tuned_thresholds = {}

for model_name in MODEL_NAMES:
    proba_cal = fitted_pipelines[model_name].predict_proba(X_cal)[:, 1]
    thr, macro_f1 = tune_threshold(y_cal, proba_cal)
    tuned_thresholds[model_name] = thr
    print(f"{model_name}: tuned threshold = {thr:.2f}  |  calibration macro-F1 = {macro_f1:.4f}")

dt: tuned threshold = 0.49  |  calibration macro-F1 = 0.5252
rf: tuned threshold = 0.51  |  calibration macro-F1 = 0.5407
gb: tuned threshold = 0.67  |  calibration macro-F1 = 0.5259
svm: tuned threshold = 0.46  |  calibration macro-F1 = 0.5352
nb: tuned threshold = 0.31  |  calibration macro-F1 = 0.5504
knn: tuned threshold = 0.47  |  calibration macro-F1 = 0.5209
xgb: tuned threshold = 0.57  |  calibration macro-F1 = 0.5374


In [8]:
results = []

for model_name in MODEL_NAMES:
    proba_test = fitted_pipelines[model_name].predict_proba(X_test)[:, 1]
    thr = tuned_thresholds[model_name]
    metrics = evaluate_at_threshold(y_test, proba_test, thr)

    auc_point, auc_lo, auc_hi = bootstrap_ci(y_test, proba_test, n_resamples=N_BOOTSTRAP, seed=SEED)

    results.append({
        "model": model_name,
        "threshold": round(thr, 2),
        "accuracy": round(metrics["accuracy"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "auc_roc": round(metrics["auc_roc"], 4),
        "auc_ci_lower": round(auc_lo, 4),
        "auc_ci_upper": round(auc_hi, 4),
        "cv_auc_tuning": round(best_cv_scores[model_name], 4),
    })
    print(f"{model_name}: Acc={metrics['accuracy']:.4f}  MacroF1={metrics['macro_f1']:.4f}  "
          f"AUC={metrics['auc_roc']:.4f} [{auc_lo:.4f}, {auc_hi:.4f}]")

results_df = pd.DataFrame(results).sort_values("auc_roc", ascending=False)
results_df

dt: Acc=0.5193  MacroF1=0.5170  AUC=0.5331 [0.5136, 0.5527]
rf: Acc=0.5486  MacroF1=0.5329  AUC=0.5571 [0.5353, 0.5792]
gb: Acc=0.5543  MacroF1=0.5067  AUC=0.5289 [0.5063, 0.5505]
svm: Acc=0.5565  MacroF1=0.5369  AUC=0.5621 [0.5391, 0.5841]
nb: Acc=0.5200  MacroF1=0.5054  AUC=0.5219 [0.5000, 0.5441]
knn: Acc=0.5679  MacroF1=0.5252  AUC=0.5312 [0.5092, 0.5528]
xgb: Acc=0.5758  MacroF1=0.5477  AUC=0.5650 [0.5433, 0.5875]


,model,threshold,accuracy,macro_f1,auc_roc,auc_ci_lower,auc_ci_upper,cv_auc_tuning
6,xgb,0.57,0.5758,0.5477,0.5650,0.5433,0.5875,0.5380
3,svm,0.46,0.5565,0.5369,0.5621,0.5391,0.5841,0.5452
1,rf,0.51,0.5486,0.5329,0.5571,0.5353,0.5792,0.5378
0,dt,0.49,0.5193,0.5170,0.5331,0.5136,0.5527,0.5294
5,knn,0.47,0.5679,0.5252,0.5312,0.5092,0.5528,0.5288
2,gb,0.67,0.5543,0.5067,0.5289,0.5063,0.5505,0.5415
4,nb,0.31,0.5200,0.5054,0.5219,0.5000,0.5441,0.5276


In [9]:
results_df.to_csv(os.path.join(TABLES_DIR, "pakistan_base_learner_results.csv"), index=False)
print(f"Saved: {os.path.join(TABLES_DIR, 'pakistan_base_learner_results.csv')}")

for model_name, pipe in fitted_pipelines.items():
    joblib.dump(pipe, os.path.join(MODELS_DIR, f"{model_name}_pipeline.joblib"))
print(f"Saved 7 fitted pipelines to {MODELS_DIR}")

Saved: C:\maternal-mental-health-risk\reports/tables\pakistan_base_learner_results.csv
Saved 7 fitted pipelines to C:\maternal-mental-health-risk\models\pakistan


In [10]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import cross_val_predict
oof_predictions = pd.DataFrame({"y_true": y_train.values}, index=X_train.index)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
for model_name in MODEL_NAMES:
    pipe = build_model_pipeline(model_name, best_params[model_name],
                                 NUMERIC_COLS, CATEGORICAL_COLS, seed=SEED)
    oof_proba = cross_val_predict(pipe, X_train, y_train, cv=cv, groups=groups_train,
                                   method="predict_proba")[:, 1]
    oof_predictions[model_name] = oof_proba
    print(f"OOF done: {model_name}")

oof_predictions.to_csv(os.path.join(data_dir, "oof_predictions_train.csv"), index=False)

OOF done: dt
OOF done: rf
OOF done: gb


C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Anaconda\envs\matmh\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `Calibra

OOF done: svm
OOF done: nb
OOF done: knn
OOF done: xgb


In [11]:
cal_predictions = pd.DataFrame({"y_true": y_cal.values})
test_predictions = pd.DataFrame({"y_true": y_test.values})

for model_name in MODEL_NAMES:
    cal_predictions[model_name] = fitted_pipelines[model_name].predict_proba(X_cal)[:, 1]
    test_predictions[model_name] = fitted_pipelines[model_name].predict_proba(X_test)[:, 1]

cal_predictions.to_csv(os.path.join(data_dir, "base_predictions_calibration.csv"), index=False)
test_predictions.to_csv(os.path.join(data_dir, "base_predictions_test.csv"), index=False)
print("Saved calibration and test base-learner predictions.")

Saved calibration and test base-learner predictions.


In [12]:
corr_matrix = oof_predictions[MODEL_NAMES].corr()
print(corr_matrix.round(3))
print("\nHigh correlation between any two tree-based models here foreshadows the same")
print("NNLS-over-logistic-regression justification used in the reference manuscript.")

        dt     rf     gb    svm     nb    knn    xgb
dt   1.000  0.738  0.122  0.748  0.422  0.171  0.525
rf   0.738  1.000  0.225  0.839  0.670  0.279  0.810
gb   0.122  0.225  1.000  0.146  0.170  0.468  0.392
svm  0.748  0.839  0.146  1.000  0.492  0.190  0.583
nb   0.422  0.670  0.170  0.492  1.000  0.171  0.639
knn  0.171  0.279  0.468  0.190  0.171  1.000  0.366
xgb  0.525  0.810  0.392  0.583  0.639  0.366  1.000

High correlation between any two tree-based models here foreshadows the same
NNLS-over-logistic-regression justification used in the reference manuscript.
